# SenseMem Best Practices

先存入真实音视频，用语义和时间找回原片，再根据记忆回答问题。
接着让音视频按时间流入 SenseMem，观察窗口如何聚合成事件，并检索、播放完整事件。
最后演示语音问题的提前召回。

从 `examples/` 目录打开 notebook，填写 API 配置后按顺序运行。
素材已放在 [media](media/README.md) 中，运行结果直接显示在每段代码下方。

## 1. 创建记忆引擎

本地嵌入模型负责存储与检索，音视频生成模型负责问答和事件切分。
填写 API 地址、Key 和模型名称；本次演示使用独立的本地工作目录。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from sensemem import Memory, build_memory_composition

In [ ]:
API_KEY = "在这里填写 API Key"
BASE_URL = "https://your-provider.example/v1"
MODEL = "qwen3.8-omni-flash"

workspace = TemporaryDirectory()
composition = build_memory_composition(
    {
        "data_dir": workspace.name,
        "embedding": {
            "provider": "sentence-transformers",
            "model": "google/embeddinggemma-2",
            "batch_size": 2,
        },
        "generation": {
            "provider": "openai",
            "base_url": BASE_URL,
            "api_key": API_KEY,
            "model": MODEL,
            "modalities": ["text", "image", "audio", "video"],
            "temperature": 0,
            "extra_body": {"reasoning_effort": "none"},
        },
    }
)
memory = Memory.from_plugins(
    composition.data_dir, plugins=composition.plugins, settings=composition.settings
)

## 2. 作为离线记忆引擎：存入与召回

存入 11 秒演讲录音和 52 秒电影预告片。
用同一天的 09:00 和 15:00 作为示例发生时间，以下时间均为 UTC。

In [ ]:
from datetime import datetime, timedelta, timezone

media = Path("media")
speech_at = datetime(2026, 11, 16, 9, tzinfo=timezone.utc)
video_at = speech_at + timedelta(hours=6)

memory.add_many(
    [media / "jfk.wav", media / "sintel.mp4"],
    occurred_at=[speech_at, video_at],
    occurred_end=[speech_at + timedelta(seconds=11), video_at + timedelta(seconds=52.209)],
);

### 用文字找回录音

In [ ]:
from IPython.display import Audio, display

hit = memory.search("哪段录音提到公民应当为国家做什么?", limit=1).hits[0]
audio = memory.read_asset(hit.id, hit.assets[0].id)
display(Audio(audio.data))

### 按时间和语义找回视频

只在 15:00–15:01 查找雪山与龙的画面。`occurred_from` / `occurred_until`
筛选与该范围重叠的记忆，随后直接播放召回的原片。

In [ ]:
from IPython.display import Video

hit = memory.search(
    "一个年轻女子穿越雪山并遇到了龙",
    occurred_from=video_at,
    occurred_until=video_at + timedelta(minutes=1),
    limit=1,
).hits[0]
print(f"发生时间: {hit.occurred_at:%H:%M:%S} - {hit.occurred_end:%H:%M:%S} UTC")
video = memory.read_asset(hit.id, hit.assets[0].id)
display(Video(video.data, embed=True, mimetype=video.media_type, width=640))

## 3. 根据记忆回答问题

将召回的原始音视频交给模型，得到具体回答。

In [ ]:
from IPython.display import Markdown

answer = memory.ask("演讲者希望公民为国家做什么?", limit=1)
display(Markdown(answer.answer))

In [ ]:
answer = memory.ask("视频中出现了什么动物?", limit=1)
display(Markdown(answer.answer))

## 4. 音视频边输入、边形成事件、边检索

同一段预告片还提供了 14 个按固定时长切好的[输入窗口](media/sintel-stream/)，
每个窗口都包含同步的视频和 WAV 音频。
这些是传输单位，SenseMem 的模型会把属于同一事件的连续窗口聚合成记忆。

下面先定义接收、事件处理和实时检索三个任务，再一起启动。

In [ ]:
import asyncio

from sensemem import AsyncEventStream, AsyncMemory, Blob, EventChunk, MemoryCategory

async_memory = AsyncMemory(memory)
work = Path(workspace.name)
origin = datetime.now(timezone.utc)
duration = 52.209
windows = sorted((media / "sintel-stream").glob("*.mp4"))
step = duration / len(windows)
query = "A young woman climbing snowy mountains"

events = AsyncEventStream(
    async_memory,
    segmenter=composition.plugins.answerer,
    checkpoint_path=work / "events.json",
    max_chunks=8,
)


def format_relative_interval(start: datetime, end: datetime) -> str:
    return f"[{(start - origin).total_seconds():.2f}, {(end - origin).total_seconds():.2f}) 秒"

### 接收带时间的窗口

`append()` 接收音视频和它的发生时间。每隔半秒送入一个窗口，模拟加速回放。

In [ ]:
records = []
stats = {"windows": 0, "searches": 0}
source_done, finished = asyncio.Event(), asyncio.Event()


async def append_video_windows() -> None:
    for index, video in enumerate(windows):
        audio = video.with_suffix(".wav")
        chunk = EventChunk(
            (Blob(video.read_bytes(), "video/mp4"), Blob(audio.read_bytes(), "audio/wav")),
            occurred_at=origin + timedelta(seconds=duration * index / len(windows)),
            occurred_end=origin + timedelta(seconds=duration * (index + 1) / len(windows)),
            stream_id="sintel",
        )
        await events.append(chunk)
        stats["windows"] += 1
        print(
            f"输入窗口 {stats['windows']}: {format_relative_interval(chunk.occurred_at, chunk.occurred_end)}"
        )
        await asyncio.sleep(0.5)
    source_done.set()

### 将连续窗口聚合成完整事件

`process()` 识别事件边界，将模型确认完整的事件存成可检索的记忆。
输入结束后，再处理最后的窗口。

In [ ]:
async def process_pending_events() -> None:
    while True:
        eof = source_done.is_set()
        async for commit in events.process(end_of_stream=eof):
            record = commit.record
            records.append(record)
            print(
                f"事件 {len(records)} 已可检索: {format_relative_interval(record.occurred_at, record.occurred_end)}"
            )
        if eof:
            finished.set()
            return
        await asyncio.sleep(0.2)

### 持续检索正在形成的记忆

这里只查 `EPISODIC` 事件记忆。首次事件提交后即可召回，其余事件继续处理。

In [ ]:
async def show_live_recall() -> None:
    previous = None
    while not finished.is_set():
        hits = (await async_memory.search(query, memory_type=MemoryCategory.EPISODIC, limit=3)).hits
        stats["searches"] += 1
        times = [format_relative_interval(hit.occurred_at, hit.occurred_end) for hit in hits]
        if times != previous:
            print(f"实时召回: {times}")
            previous = times
        await asyncio.sleep(1)

### 启动三个并发任务

下方日志直接展示本次运行的输入、事件提交和召回顺序。

In [ ]:
async with asyncio.TaskGroup() as tasks:
    tasks.create_task(append_video_windows())
    tasks.create_task(process_pending_events())
    tasks.create_task(show_live_recall())

### 看窗口如何聚合成事件

表格来自本次提交的事件，展示每个事件包含的窗口和发生时间。事件边界由模型判断。

In [ ]:
numbers = {record.id: number for number, record in enumerate(records, 1)}
rows = ["| 事件 | 输入窗口 | 发生时间 (相对素材起点) |", "| --- | --- | --- |"]
for number, record in enumerate(records, 1):
    chunks = record.metadata["event_segmentation"]["chunks"]
    first = round((record.occurred_at - origin).total_seconds() / step) + 1
    last = first + len(chunks) - 1
    rows.append(
        f"| {number} | {first}-{last}, 共 {len(chunks)} 个窗口 "
        f"| {format_relative_interval(record.occurred_at, record.occurred_end)} |"
    )
display(Markdown("\n".join(rows)))
print(
    f"接收 {stats['windows']} 个窗口, 形成 {len(records)} 个事件; 并发检索 {stats['searches']} 次。"
)

### 按语义和时间检索事件

先查素材前半段，再查素材开始前的空白时间；最后用第一个事件的实际结束时间测试边界。
时间范围为左闭右开，返回与它重叠的完整事件。

In [ ]:
first_event = records[0]
boundary = first_event.occurred_end
summary = first_event.metadata["event_segmentation"]["summary"]
queries = [
    ("雪山中的年轻女子", query, origin, origin + timedelta(seconds=duration / 2)),
    ("相同语义: 素材开始前", query, origin - timedelta(seconds=2), origin - timedelta(seconds=1)),
    ("事件 1 的摘要: 结束前", summary, boundary - timedelta(seconds=0.25), boundary),
    ("事件 1 的摘要: 从结束开始", summary, boundary, boundary + timedelta(seconds=0.25)),
]
retrieved = []
rows = ["| 查询 | 时间范围 (右端不含) | 实际召回 |", "| --- | --- | --- |"]
for label, text, start, end in queries:
    result = await async_memory.search(
        text,
        memory_type=MemoryCategory.EPISODIC,
        occurred_from=start,
        occurred_until=end,
        limit=10,
    )
    retrieved.append(result.hits)
    matches = "、".join(f"事件 {numbers[hit.id]}" for hit in result.hits) or "无匹配事件"
    rows.append(f"| {label} | {format_relative_interval(start, end)} | {matches} |")
display(Markdown("\n".join(rows)))

### 播放刚刚召回的完整事件

用 `read_asset()` 读回事件中的原视频窗口，再用 `ffmpeg` 按顺序拼接，保留同步音轨。
下面展示前半段查询实际命中的事件摘要、时间和音视频。

In [ ]:
import subprocess

for hit in retrieved[0]:
    summary = hit.metadata["event_segmentation"]["summary"]
    display(
        Markdown(
            f"### 事件 {numbers[hit.id]}: {format_relative_interval(hit.occurred_at, hit.occurred_end)}\n\n{summary}"
        )
    )

    files = []
    for index, chunk in enumerate(hit.metadata["event_segmentation"]["chunks"]):
        video = await async_memory.read_asset(hit.id, chunk["asset_ids"][0])
        path = work / f"window-{index}.mp4"
        path.write_bytes(video.data)
        files.append(f"file '{path}'")
    playlist = work / "event.txt"
    playlist.write_text("\n".join(files))
    output = work / "event.mp4"
    command = ["ffmpeg", "-v", "error", "-y", "-f", "concat", "-safe", "0"]
    command += ["-i", str(playlist), "-c", "copy", str(output)]
    subprocess.run(command, check=True, capture_output=True)
    display(Video(filename=str(output), embed=True, width=640))

### 根据刚形成的事件回答问题

In [ ]:
answer = await async_memory.ask(
    "Which animal appears in the recording? Answer with the animal name only.",
    memory_type=MemoryCategory.EPISODIC,
    limit=5,
)
display(Markdown(answer.answer))

## 5. 进阶：问题没说完，提前召回

把 ASR 的中间文字交给 `submit()`，说完后用 `finalize()` 获取最终问题的结果。

In [ ]:
from sensemem import AsyncOmniPrefetch

prefetch = AsyncOmniPrefetch(async_memory, limit=1)
prefetch.submit("哪段录音提到公民")
await asyncio.sleep(0.3)  # 模拟继续说话的间隔
result = await prefetch.finalize("哪段录音提到公民应当为国家做什么?")
hit = result.hits[0]
audio = await async_memory.read_asset(hit.id, hit.assets[0].id)
display(Audio(audio.data))

### 结束示例

关闭实例，清理本次演示的工作目录。音视频播放器已嵌入 notebook 输出中。
素材来源与许可证见 [media/README.md](media/README.md)。

In [ ]:
import gc

await async_memory.close()
workspace.cleanup()
events = prefetch = async_memory = memory = composition = None
_ = gc.collect()